In [8]:
import pandas as pd

# Cargar datos
df_article_trasactions = pd.read_csv('df_reviews_articles_v_reducido.csv')
df_article_trasactions.head()

,t_dat,customer_id,article_id,price,sales_channel_id,count,popularity_score,p_review,review_flag,product_code,...,department_name,index_code,index_name,index_group_no,index_group_name,section_no,section_name,garment_group_no,garment_group_name,detail_desc
0,2019-02-25,00f6a35b0c389c0f64d081b64ac98dd38dbc0e6a51ab66...,108775015,0.008458,2,10841,0.000341,0.03,True,108775,...,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.
1,2018-12-23,fbbc4b14371dac97483160a35dbd93e7a57e292aedbe2c...,108775015,0.007186,2,10841,0.000341,0.03,True,108775,...,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.
2,2019-02-03,4c53008e64aa6c59bf1a2b7025ae4afb98be8bc8b457c0...,108775015,0.008458,2,10841,0.000341,0.03,True,108775,...,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.
3,2018-12-12,6bdaa2c45d8f21f24bc42f62b873897dec4e5cc00a69cf...,108775015,0.008458,1,10841,0.000341,0.03,True,108775,...,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.
4,2018-12-03,1033afaf7b151c626d26baa254b851a2d15368b43b215e...,108775015,0.008034,1,10841,0.000341,0.03,True,108775,...,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.


In [9]:
SYSTEM_PROMPT = """
You generate synthetic product reviews based on tabular product attributes and the number of stars.

Your objectives:
- Write natural, human-like reviews that sound like real customers.
- Use product attributes (name, type, appearance, color, category, description, etc.) but never list them explicitly.
- You may include personal context or small anecdotes that make the review feel authentic.
- Never mention dataset columns, IDs, internal variables or anything technical.
- Only return the review text.

Tone rules:
- 5 stars → enthusiastic, descriptive, clearly positive.
- 4 stars → positive with 1–2 small imperfections.
- 3 stars → neutral or mixed experience.
- 2 stars → mostly negative but still reasonable.
- 1 star → short, direct, clearly dissatisfied.

Length rules:
- If user mentions `{short review}`, keep it brief (1–2 sentences).
- If user mentions `{long review}`, write a longer narrative (1–3 paragraphs).
- Otherwise choose a natural length.

Now follow the examples below:

Example A — 5 stars (long, very positive)
article: Lee Extreme Motion Pants
5 stars

I’ve tried so many work pants over the years, but these are easily the best. The stretch is perfect—enough give without feeling loose—and the black color goes with everything. What surprised me most is how consistent the fit is across colors. Other brands change sizing depending on dye, but not these.

I climb ladders, kneel, crawl under equipment… they move with me without digging into my calves. After months of constant washing they still haven’t shrunk or faded. Honestly the most reliable pants I own.

Example B — 4 stars (long, balanced)
article: Lee Work Pants
4 stars

As a crane service tech, my clothes take a beating. These pants are lightweight, flexible, and much easier to clean than I expected. They fit comfortably around the waist and the stretch in the back keeps them from feeling restrictive.

The downside is the sizing. A 36×32 feels a little loose, but the 36×30 is uncomfortably tight. And the navy blue is really more of a charcoal gray and fades quickly. Still, for the price and durability, they’re great work pants.

Example C — 3 stars (short)
article: Lee Casual Pants
3 stars

Comfortable enough for daily wear, but the fit from the knee down feels off. Even after ironing they never look completely straight.

Example D — 2 stars (short negative)
article: Summer Strap Top
2 stars

The material is soft but way too thin, and the straps don’t stay in place. Feels cheap for what it costs.

Example E — 1 star (very short, very negative)
article: Basic Tank Top
1 star

Poor quality. Looks stretched out after a single wash.

"""


In [10]:
import pandas as pd
from groq import Groq
import numpy as np
import re
import time
from typing import List

client = Groq(api_key="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")
LINE_PATTERN = re.compile(r"^\s*(\d+)\s*\)\s*(.+)$", re.MULTILINE)


def assign_star_distribution(percentile):
    if percentile >= 0.80:  # top 20%
        return np.random.choice([5, 4, 3, 2, 1], p=[0.70, 0.20, 0.05, 0.03, 0.02])
    if percentile >= 0.20:  # media
        return np.random.choice([5, 4, 3, 2, 1], p=[0.60, 0.20, 0.10, 0.05, 0.05])
    return np.random.choice([5, 4, 3, 2, 1], p=[0.50, 0.20, 0.15, 0.10, 0.05])


def safe_value(row: pd.Series, column: str) -> str:
    """Return a clean string value for prompt construction."""
    value = row.get(column, "N/A")
    if pd.isna(value):
        return "N/A"
    return str(value)


def build_batch_prompt(rows: List[pd.Series]) -> str:
    """Format the batch into a single numbered prompt for Groq."""
    prompt_lines = [
        "You will generate ONE independent review for EACH item below.",
        "",
        "Rules:",
        "- Treat each item independently.",
        "- Do NOT mix information between items.",
        "- For each item i, write exactly ONE review line starting with the item number.",
        "- Only return the reviews, nothing else.",
        "",
        "Items:",
    ]

    for idx, row in enumerate(rows, start=1):
        prompt_lines.append(
            (
                f"{idx}) stars: {safe_value(row, 'review_stars')}, "
                f"product name: {safe_value(row, 'prod_name')}, "
                f"product type: {safe_value(row, 'product_type_name')}, "
                f"product group: {safe_value(row, 'product_group_name')}, "
                f"appearance: {safe_value(row, 'graphical_appearance_name')}, "
                f"color: {safe_value(row, 'colour_group_name')}, "
                f"department: {safe_value(row, 'department_name')}, "
                f"section: {safe_value(row, 'section_name')}, "
                f"description: {safe_value(row, 'detail_desc')}, "
                f"price: {safe_value(row, 'price')}, "
                f"sales channel: {safe_value(row, 'sales_channel_id')}"
            )
        )

    prompt_lines.extend(
        [
            "",
            "Output format:",
            "For each item i return exactly one line in this format:",
            "i) <review for item i>",
        ]
    )
    return "\n".join(prompt_lines)


def parse_batch_response(message: str, expected_count: int) -> List[str]:
    """Parse numbered reviews from the model response and keep ordering."""
    matches = LINE_PATTERN.findall(message)
    reviews = ["" for _ in range(expected_count)]

    for idx_str, review_text in matches:
        idx = int(idx_str)
        if 1 <= idx <= expected_count and not reviews[idx - 1]:
            reviews[idx - 1] = review_text.strip()

    if any(not review for review in reviews):
        raise ValueError(
            f"Incomplete batch response. Expected {expected_count} items, got: {message}"
        )
    return reviews


def generate_reviews_batch(
    rows: List[pd.Series], max_retries: int = 3, backoff_seconds: float = 2.0
) -> List[str]:
    """Call Groq once per batch and return reviews preserving original order."""
    if not rows:
        return []

    prompt = build_batch_prompt(rows)
    for attempt in range(1, max_retries + 1):
        try:
            completion = client.chat.completions.create(
                model="meta-llama/llama-4-maverick-17b-128e-instruct",
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": prompt + "\n\nNO EXTRAS. SOLO LÍNEAS NUMERADAS."},
                ],
                temperature=0.7,
                max_completion_tokens=1200,
            )
            content = completion.choices[0].message.content.strip()
            return parse_batch_response(content, expected_count=len(rows))
        except Exception as exc:
            if attempt == max_retries:
                raise
            wait_time = backoff_seconds * attempt
            print(
                f"Batch generation failed (attempt {attempt}/{max_retries}). Retrying in {wait_time:.1f}s..."
            )
            print(f"Error: {exc}")
            time.sleep(wait_time)

    raise RuntimeError("Unable to generate batch reviews after retries.")


In [11]:
df = df_article_trasactions.copy()
df["pop_rank"] = df["popularity_score"].rank(pct=True)
df["review_stars"] = df["pop_rank"].apply(assign_star_distribution)
df.head()


,t_dat,customer_id,article_id,price,sales_channel_id,count,popularity_score,p_review,review_flag,product_code,...,index_name,index_group_no,index_group_name,section_no,section_name,garment_group_no,garment_group_name,detail_desc,pop_rank,review_stars
0,2019-02-25,00f6a35b0c389c0f64d081b64ac98dd38dbc0e6a51ab66...,108775015,0.008458,2,10841,0.000341,0.03,True,108775,...,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.,0.942985,5
1,2018-12-23,fbbc4b14371dac97483160a35dbd93e7a57e292aedbe2c...,108775015,0.007186,2,10841,0.000341,0.03,True,108775,...,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.,0.942985,5
2,2019-02-03,4c53008e64aa6c59bf1a2b7025ae4afb98be8bc8b457c0...,108775015,0.008458,2,10841,0.000341,0.03,True,108775,...,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.,0.942985,5
3,2018-12-12,6bdaa2c45d8f21f24bc42f62b873897dec4e5cc00a69cf...,108775015,0.008458,1,10841,0.000341,0.03,True,108775,...,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.,0.942985,4
4,2018-12-03,1033afaf7b151c626d26baa254b851a2d15368b43b215e...,108775015,0.008034,1,10841,0.000341,0.03,True,108775,...,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.,0.942985,5


In [12]:
df['review_stars'].value_counts().sort_index()

review_stars
1     2598
2     3328
3     5852
4    11901
5    35779
Name: count, dtype: int64

In [13]:
df_article_trasactions = pd.read_csv('df_reviews_articles_v_reducido.csv')
df = df_article_trasactions.copy()
df['pop_rank'] = df['popularity_score'].rank(pct=True)
df['review_stars'] = df['pop_rank'].apply(assign_star_distribution)

# Prepare container for generated reviews.
df_with_reviews = df.copy()
df_with_reviews['review'] = pd.Series([None] * len(df_with_reviews), dtype='object')
BATCH_SIZE = 8
print(
    f"Dataset ready with {len(df_with_reviews)} rows. Generating reviews in batches of {BATCH_SIZE}."
)


Dataset ready with 59458 rows. Generating reviews in batches of 8.


In [14]:
for start in range(0, len(df_with_reviews), BATCH_SIZE):
    batch_df = df_with_reviews.iloc[start : start + BATCH_SIZE]
    rows_to_review = batch_df[batch_df['review_flag'].fillna(False)]

    if rows_to_review.empty:
        continue

    idxs = rows_to_review.index.tolist()  # Keep original DataFrame indexes.
    rows = [row for _, row in rows_to_review.iterrows()]
    reviews = generate_reviews_batch(rows)
    df_with_reviews.loc[idxs, 'review'] = reviews

    print(
        f"Processed batch rows {start}-{start + len(batch_df) - 1}: generated {len(reviews)} reviews."
    )

df_with_reviews[['prod_name', 'review_stars', 'review_flag', 'review']].head(10)


Processed batch rows 0-7: generated 8 reviews.
Processed batch rows 8-15: generated 8 reviews.
Processed batch rows 16-23: generated 8 reviews.
Processed batch rows 24-31: generated 8 reviews.
Processed batch rows 32-39: generated 8 reviews.
Processed batch rows 40-47: generated 8 reviews.
Processed batch rows 48-55: generated 8 reviews.
Processed batch rows 56-63: generated 8 reviews.
Processed batch rows 64-71: generated 8 reviews.
Processed batch rows 72-79: generated 8 reviews.
Processed batch rows 80-87: generated 8 reviews.
Processed batch rows 88-95: generated 8 reviews.
Processed batch rows 96-103: generated 8 reviews.
Processed batch rows 104-111: generated 8 reviews.
Processed batch rows 112-119: generated 8 reviews.
Processed batch rows 120-127: generated 8 reviews.
Processed batch rows 128-135: generated 8 reviews.
Processed batch rows 136-143: generated 8 reviews.
Processed batch rows 144-151: generated 8 reviews.
Processed batch rows 152-159: generated 8 reviews.
Processe

,prod_name,review_stars,review_flag,review
0,Strap top,3,True,"It's an okay top, the straps are a bit thin bu..."
1,Strap top,4,True,"I like this top, it's simple and comfy, the bl..."
2,Strap top,4,True,"Great value for the price, this strap top is s..."
3,Strap top,5,True,"I'm obsessed with this strap top, it's so vers..."
4,Strap top,5,True,"This strap top is a staple in my wardrobe, it'..."
5,Strap top,5,True,"I've bought this top multiple times, it's a gr..."
6,Strap top,5,True,"I love this simple yet stylish top, the narrow..."
7,Strap top,5,True,"What a great find, this strap top is so afford..."
8,Strap top,5,True,I'm obsessed with this simple yet stylish stra...
9,Strap top,5,True,I've bought this top multiple times because it...


In [17]:
df_with_reviews.to_csv("df_reviews_articles_with_synth_reviews.csv", index=False)


In [18]:
df_with_reviews[['prod_name', 'review_stars', 'review_flag', 'review']].head(10)


,prod_name,review_stars,review_flag,review
0,Strap top,3,True,"It's an okay top, the straps are a bit thin bu..."
1,Strap top,4,True,"I like this top, it's simple and comfy, the bl..."
2,Strap top,4,True,"Great value for the price, this strap top is s..."
3,Strap top,5,True,"I'm obsessed with this strap top, it's so vers..."
4,Strap top,5,True,"This strap top is a staple in my wardrobe, it'..."
5,Strap top,5,True,"I've bought this top multiple times, it's a gr..."
6,Strap top,5,True,"I love this simple yet stylish top, the narrow..."
7,Strap top,5,True,"What a great find, this strap top is so afford..."
8,Strap top,5,True,I'm obsessed with this simple yet stylish stra...
9,Strap top,5,True,I've bought this top multiple times because it...


In [ ]:
df_with_reviews.value_counts()